In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import sys
import logging
import os
import random
import time
import re
import math
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from bs4 import BeautifulSoup
from dateutil import parser as date_parser

# --- Selenium ---
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager

# --- Requests ---
import requests
import warnings
from urllib3.exceptions import InsecureRequestWarning

warnings.simplefilter('ignore', InsecureRequestWarning)

# ===============================================================
#  1. 全域設定
# ===============================================================

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

DATA_FILENAME = os.path.join(BASE_DIR, 'News_Data.csv')
LOG_FILENAME = os.path.join(BASE_DIR, 'scraper_ltn_test.log')
V5_COLUMNS = ['url', 'media', 'category', 'publish_time', 'title', 'content', 'fetch_time', 'label']

# 測試用數量
LIMIT_L1 = 50
LIMIT_L0 = 75

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILENAME, encoding='utf-8', mode='a'),
        logging.StreamHandler(sys.stdout)
    ]
)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Referer': 'https://news.ltn.com.tw/'
}

# ===============================================================
#  2. LTN 專用設定 (含全子網域支援)
# ===============================================================

LTN_CONFIG = {
    'name': 'LTN',
    'hot_url': 'https://news.ltn.com.tw/list/breakingnews/popular',
    'latest_url': 'https://news.ltn.com.tw/list/breakingnews',
    'base_url': 'https://news.ltn.com.tw',
    # 廣域選擇器：抓取容器內所有連結，後續再過濾
    'hot_selector': ['div.whitecon a', 'ul.list a', 'div.boxTitle a', 'a.tit'],
    'latest_selector': ['div.whitecon ul.list li a', 'div.whitecon a', 'ul.list a']
}

# 內文選擇器 (涵蓋 LTN 主站/體育/娛樂/財經/汽車)
CONTENT_SELECTORS = [
    'div[itemprop="articleBody"]',  # LTN 主站標準
    'div.text',                     # LTN 財經/娛樂/舊版/通用
    'div.news_content',             # LTN 體育
    'div.box_data',                 # LTN 汽車/部分子版
    'div.article-body',             # 通用備案
    'div.content',
    'article'
]

# ===============================================================
#  3. 資料清洗與中文優化模組 (V13.1)
# ===============================================================

# 翻譯字典 (備案用，當麵包屑抓不到時才用)
CATEGORY_MAP = {
    'politics': '政治', 'society': '社會', 'life': '生活', 'world': '國際',
    'local': '地方', 'novelty': '搜奇', 'business': '財經', 'finance': '財經',
    'ec': '財經', 'sports': '體育', 'entertainment': '娛樂', 'ent': '娛樂',
    'star': '娛樂', 'health': '健康', 'supplement': '副刊', 'opinion': '言論',
    'talk': '言論', 'focus': '焦點', '3c': '3C科技', 'auto': '汽車', 'fashion': '時尚'
}

def clean_category(cat_str):
    """清洗並翻譯分類"""
    if not cat_str: return "即時"
    
    cat_str = str(cat_str).strip()
    
    # 處理麵包屑格式 (例如: "首頁 > 政治" 或 "Home / Politics")
    if '>' in cat_str: 
        parts = cat_str.split('>')
        # 取倒數第二個通常是主分類 (最後一個可能是標題)
        # 例如: 首頁 > 政治 > 立委選舉... -> 取 "政治"
        if len(parts) >= 2:
            candidate = parts[-1].strip()
            # 如果最後一段太長(>10字)，很可能是標題，改取上一層
            if len(candidate) > 10:
                cat_str = parts[-2].strip()
            else:
                cat_str = candidate
        else:
            cat_str = parts[-1].strip()
            
    elif '/' in cat_str and len(cat_str) < 20: 
        cat_str = cat_str.split('/')[-1].strip()

    # 去除 "LTN" 字樣
    cat_str = cat_str.replace("自由時報", "").replace("LTN", "").strip()

    # 查表翻譯 (如果抓下來還是英文，例如 Meta Tag)
    lower_cat = cat_str.lower()
    if lower_cat in CATEGORY_MAP:
        return CATEGORY_MAP[lower_cat]
        
    return cat_str

def format_time_strict(time_str):
    now = datetime.now()
    if not time_str or str(time_str).lower() in ['nan', 'n/a', 'none', '']:
        return now.strftime("%Y/%-m/%-d %-I:%M:%S %p")
    try:
        raw = str(time_str).replace("發布時間：", "").replace("更新時間：", "").strip()
        if "小時前" in raw:
            hours = int(re.search(r'(\d+)', raw).group(1))
            dt = now - timedelta(hours=hours)
        elif "分鐘前" in raw:
            mins = int(re.search(r'(\d+)', raw).group(1))
            dt = now - timedelta(minutes=mins)
        elif "剛剛" in raw:
            dt = now
        else:
            dt = date_parser.parse(raw, fuzzy=True)
        
        hour_12 = int(dt.strftime("%I"))
        time_part = f"{hour_12}:{dt.strftime('%M:%S %p')}"
        return f"{dt.year}/{dt.month}/{dt.day} {time_part}"
    except:
        return now.strftime("%Y/%-m/%-d %-I:%M:%S %p")

def parse_html_content(html_text):
    soup = BeautifulSoup(html_text, 'lxml')
    content = np.nan
    
    # 1. 內文
    for sel in CONTENT_SELECTORS:
        elm = soup.select_one(sel)
        if elm:
            for tag in elm(['script', 'style', 'iframe', 'div.knn_related', 'div.fb-root', 'div.author', 'div.ad', 'div.click_mic', 'p.app_ad']):
                tag.decompose()
            text = elm.get_text(separator=' ', strip=True)
            text = re.sub(r'\s+', ' ', text)
            if len(text) > 30:
                content = text
                break
    
    # [V13.1 核心修正] 優先抓取麵包屑 (Breadcrumbs)
    category = None
    
    # LTN 各種子網域的麵包屑選擇器
    crumbs_selectors = [
        'div.guide',            # news 主站
        'div.breadcrumbs',      # sports, ent
        'div.breadcrumb',       # ec, auto
        'nav[aria-label="breadcrumb"]',
        'div.path'
    ]
    
    for sel in crumbs_selectors:
        crumbs = soup.select_one(sel)
        if crumbs:
            # 取純文字，例如 "首頁 > 政治 > ..."
            category = crumbs.get_text(separator=' > ', strip=True)
            break
    
    # 如果麵包屑失敗，才抓 Meta Tag (通常是英文)
    if not category:
        meta_sec = soup.select_one('meta[property="article:section"]')
        if meta_sec and meta_sec.get('content'):
            category = meta_sec.get('content')
            
    if not category:
        category = "即時"

    # 時間
    raw_time = None
    time_elm = soup.select_one('meta[property="article:published_time"], meta[name="pubdate"], time, span.time, div.date, div.meta-info time')
    if time_elm:
        raw_time = time_elm.get('content') if time_elm.has_attr('content') else time_elm.get_text(strip=True)
    
    return clean_category(category), format_time_strict(raw_time), content

# ===============================================================
#  4. 爬蟲核心
# ===============================================================

class HumanNavigator:
    def __init__(self, driver):
        self.driver = driver
        self.action = ActionChains(driver)

    def random_sleep(self, min_s=1.5, max_s=3.0):
        time.sleep(random.uniform(min_s, max_s))

    def smooth_scroll(self):
        try:
            current_pos = self.driver.execute_script("return window.pageYOffset;")
            target_distance = random.randint(800, 1200)
            steps = random.randint(10, 20)
            for _ in range(steps):
                move = (target_distance / steps) + random.uniform(-5, 5) 
                current_pos += move
                self.driver.execute_script(f"window.scrollTo(0, {current_pos});")
                time.sleep(0.02)
        except: pass

class NewsSpider:
    def __init__(self):
        self.options = Options()
        self.options.add_argument("--headless")
        self.options.add_argument("--disable-gpu")
        self.options.add_argument("--no-sandbox")
        self.options.add_argument("--window-size=1920,1080")
        self.options.add_argument(f"user-agent={HEADERS['User-Agent']}")
        self.options.add_argument("--disable-blink-features=AutomationControlled")
        self.options.add_experimental_option("excludeSwitches", ["enable-automation"])
        self.options.add_experimental_option('useAutomationExtension', False)
        
        prefs = {"profile.managed_default_content_settings.images": 2}
        self.options.add_experimental_option("prefs", prefs)
        
        self.driver = None
        self.human = None
        self.existing_data = self._get_existing_data()

    def _get_existing_data(self):
        existing = {}
        if os.path.exists(DATA_FILENAME):
            try:
                df = pd.read_csv(DATA_FILENAME)
                if 'url' in df.columns and 'label' in df.columns:
                    for idx, row in df.iterrows():
                        existing[str(row['url'])] = row['label']
            except: pass
        return existing

    def start(self):
        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=self.options)
        self.driver.set_page_load_timeout(45)
        self.driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        self.human = HumanNavigator(self.driver)

    def stop(self):
        if self.driver:
            try: self.driver.quit()
            except: pass
            self.driver = None

    def fetch_content_smart(self, url):
        # 1. Requests
        try:
            res = requests.get(url, headers=HEADERS, timeout=6, verify=False)
            if res.status_code == 200:
                cat, t, c = parse_html_content(res.text)
                if isinstance(c, str): return cat, t, c
        except: pass

        # 2. Selenium
        try:
            self.driver.execute_script("window.open('');")
            self.driver.switch_to.window(self.driver.window_handles[-1])
            try:
                self.driver.get(url)
            except TimeoutException:
                self.driver.execute_script("window.stop();")
            
            self.human.random_sleep(1.0, 2.0)
            html = self.driver.page_source
            self.driver.close()
            self.driver.switch_to.window(self.driver.window_handles[0])
            return parse_html_content(html)
        except:
            try:
                if len(self.driver.window_handles) > 1:
                    self.driver.close()
                    self.driver.switch_to.window(self.driver.window_handles[0])
            except: pass
        return "即時", format_time_strict(None), np.nan

    def crawl_task(self, site_config, label, limit):
        url = site_config['hot_url'] if label == 1 else site_config['latest_url']
        selectors = site_config['hot_selector'] if label == 1 else site_config['latest_selector']
        site_name = site_config['name']

        logging.info(f"      L{label} 任務啟動 (目標: {limit} 筆)...")
        try:
            try:
                self.driver.get(url)
                try:
                    WebDriverWait(self.driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
                    time.sleep(5) # LTN 需要多一點時間
                except: pass
            except TimeoutException:
                logging.warning("      ⚠️ 列表載入超時，強制停止...")
                self.driver.execute_script("window.stop();")

            collected = []
            seen_links = set()
            
            # 捲動
            scroll_target = int((limit / 8) * 1.5) + 3
            for _ in range(scroll_target):
                self.human.smooth_scroll()
                self.human.random_sleep(1.0, 1.5)
                soup = BeautifulSoup(self.driver.page_source, 'lxml')
                found_count = 0
                for sel in selectors: found_count += len(soup.select(sel))
                if found_count >= limit * 1.2: break
            
            soup = BeautifulSoup(self.driver.page_source, 'lxml')
            elements = []
            for sel in selectors:
                found = soup.select(sel)
                if found: elements.extend(found)

            logging.info(f"      -> 掃描到 {len(elements)} 個連結...")

            for item in elements:
                if len(collected) >= limit: break
                
                try:
                    link = item.get('href')
                    if not link: continue
                    # LTN 過濾器 (包含所有子網域)
                    if not any(x in link for x in ['/news/', '/article/', 'sports', 'ent', 'ec', 'auto']): continue
                    
                    title = item.get('title') or item.get_text(strip=True)
                    if not link or not title or len(title) < 4: continue
                    
                    if link.startswith('/'):
                        base = site_config['base_url']
                        full_url = base.rstrip('/') + '/' + link.lstrip('/')
                    elif link.startswith('http'):
                        full_url = link
                    else: continue
                    
                    clean_url = full_url.split('?')[0]

                    if clean_url in seen_links: continue
                    seen_links.add(clean_url)

                    # 智慧去重與升級
                    is_in_db = clean_url in self.existing_data
                    db_label = self.existing_data.get(clean_url, -1)

                    if is_in_db:
                        if label == 1 and db_label == 0:
                            logging.info(f"         [Upgrade] 發現 L0->L1 潛力股: {title[:8]}")
                        elif label == 1 and db_label == 1: continue
                        elif label == 0: continue

                    cat, t_str, content = self.fetch_content_smart(full_url)
                    if not isinstance(content, str) or len(content) < 30: continue 

                    fetch_time = datetime.now().strftime("%Y/%-m/%-d %-I:%M:%S %p")
                    
                    collected.append({
                        'url': full_url,
                        'media': site_name,
                        'category': cat, # 這裡應該要是中文了
                        'publish_time': t_str,
                        'title': title,
                        'content': content,
                        'fetch_time': fetch_time,
                        'label': label
                    })
                    
                    print(f"         [L{label}] {title[:8]}... (OK) - {cat}")
                    time.sleep(random.uniform(1.5, 3.0))

                except Exception: continue
            
            return collected

        except Exception as e:
            logging.error(f"      🔥 任務錯誤: {e}")
            return []

# ===============================================================
#  主程式 (LTN Only)
# ===============================================================

def main():
    logging.info(f"🚀 T-Forecast V13.1 (LTN Native Chinese Test) 啟動")
    
    spider = NewsSpider()
    site_config = LTN_CONFIG # 只跑 LTN
    site_name = site_config['name']
    
    logging.info(f"--- 處理媒體: {site_name} ---")
    
    spider.start()
    all_data = []
    
    try:
        # 1. 熱門 (L1)
        l1_data = spider.crawl_task(site_config, label=1, limit=LIMIT_L1)
        all_data.extend(l1_data)
        logging.info(f"      ✅ {site_name} L1 完成: {len(l1_data)} 筆")
        
        spider.human.random_sleep(2, 3)
        
        # 2. 即時 (L0)
        l0_data = spider.crawl_task(site_config, label=0, limit=LIMIT_L0)
        all_data.extend(l0_data)
        logging.info(f"      ✅ {site_name} L0 完成: {len(l0_data)} 筆")
        
    except Exception as e:
        logging.error(f"   🔥 {site_name} 發生錯誤: {e}")
    
    spider.stop()
    
    if all_data:
        df = pd.DataFrame(all_data)
        for col in V5_COLUMNS:
            if col not in df.columns: df[col] = np.nan
        df = df[V5_COLUMNS]
        try:
            df.to_csv(DATA_FILENAME, mode='a', header=False, index=False, encoding='utf-8-sig')
            logging.info(f"   💾 {site_name} 存檔成功 (共 {len(all_data)} 筆)")
        except:
            df.to_csv(f"backup_{site_name}_{int(time.time())}.csv", index=False)

    logging.info("🏁 LTN 任務結束")

if __name__ == "__main__":
    main()

2025-12-10 23:06:22,518 - INFO - 🚀 T-Forecast V13.1 (LTN Native Chinese Test) 啟動
2025-12-10 23:06:22,518 - INFO - --- 處理媒體: LTN ---
2025-12-10 23:06:22,518 - INFO - ====== WebDriver manager ======
2025-12-10 23:06:22,810 - INFO - Get LATEST chromedriver version for google-chrome
2025-12-10 23:06:23,096 - INFO - Get LATEST chromedriver version for google-chrome
2025-12-10 23:06:23,343 - INFO - Driver [/Users/xiaobing/.wdm/drivers/chromedriver/mac64/143.0.7499.40/chromedriver-mac-arm64/chromedriver] found in cache
2025-12-10 23:06:24,406 - INFO -       L1 任務啟動 (目標: 50 筆)...
2025-12-10 23:06:32,826 - INFO -       -> 掃描到 198 個連結...
         [L1] 麥當勞太貴了！菜... (OK) - 國際財經
         [L1] 憋不住了！飛機中... (OK) - 國際
         [L1] MLB》宇宙道奇... (OK) - MLB
         [L1] 「自然人憑證」翻... (OK) - 政治
         [L1] 獨家》助理費除罪... (OK) - 政治
         [L1] 日本人買下UNI... (OK) - 國際財經
         [L1] 獨家》中國報復「... (OK) - 政治
         [L1] 親中派跳腳！宏都... (OK) - 國際
         [L1] 起床別急著整理床... (OK) - 國際
         [L1] 中職》黃勇傳加盟... (OK) - 中職